## 1. torch_prep_kfold.py --initial_split  → writes *_trn_final.csv, *_tst_preprocess.csv

In [37]:
import sys
import logging
import argparse
import numpy as np
import pandas as pd
from typing import Any, Optional, List, Tuple
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
import os

In [38]:
random_state = 42
test_percentage = 0.15
np.random.seed(42)

In [39]:
###############################################################################
# Data Loading (Reference + Features)
###############################################################################

In [40]:
id_col = 'sequence'
label_col = 'bind_avg'
df1 = pd.read_csv('exp_data_all.csv')
ref_data = df1[[id_col, label_col]].copy()

print(ref_data)

                                 sequence  bind_avg
0    GTACACAATTTTTTACAAAATTTAAATTAAAACAAA -0.862667
1    GCAGCCGAGGCGGAGAGAGAGAGAGGACAGCTTACG -0.703319
2    AGGCCCAGGAAGAACAATGGCTCTGCCAACTGGGCA -0.659464
3    CTTCCTCACCTGCAGACTTCCTTCCCTGAGTCCCAG -0.543823
4    TGAGGGTCAGAGGCACCCCTTCCTGGAATCTCCTTC -0.424828
..                                    ...       ...
163  ATCTCCTGGGGCGACCACGAGGTCACCCGTCCAGGT  1.524243
164  GAAAACCAGCGAGACCGCATGGTCTCACTTATAAGT  1.452711
165  AGGGAGTTCTCACACCATGTGGGTGGGATTGTAACT  1.305160
166  ACACTGAGCTTCCTCCACGTGCCCAGGTCCTGGCAG  1.430431
167  GTGTCTCCATTGGGGCACGTGTTTATATGTTTATAA  1.598717

[168 rows x 2 columns]


In [41]:
usecols = ['sequence','run','VDWAALS','EEL','EGB','ESURF','HB Energy','Hydrophobic Energy','Pi-Pi Energy','Delta_Entropy']

df2 = pd.read_csv('rawdat.csv', usecols=usecols)
feature_data = df2.copy()

print(feature_data)

                                    sequence  run  VDWAALS       EEL  \
0       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -252.110 -1886.830   
1       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -238.510 -1881.424   
2       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -246.721 -1895.687   
3       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -235.671 -1857.573   
4       GCGCTGGGAGGCGAACACGTGCCCGCCGCCCCATCC    9 -230.214 -1897.268   
...                                      ...  ...      ...       ...   
272155  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -148.291 -1941.978   
272156  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -142.375 -1959.709   
272157  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -167.433 -1911.650   
272158  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -136.663 -1920.439   
272159  TTTTTTTTTTTTTTGAGAAAATGAAGACAATTATCT   20 -146.999 -1929.330   

             EGB   ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  \
0       1841.253 -36.482  -1.940432         -165.447020 -1.655

In [42]:
## I am not doing sequence level scrambling

1) --initial_split:
   - Merges feature data (optionally from multiple files) with reference data using a shared ID column.
   - Performs an optional "sequence-level scrambling" of labels in the TRAINING set only, controlled by `scramble_fractions`.
   - Splits into train and test sets by unique sequence ID (or stratified for classification).
   - Saves the resulting CSV files:
       * PREFIX_MODELTYPE_scrFRAC_trn_final.csv   (training)
       * PREFIXMODELTYPE_scrFRAC_tst_preprocess.csv (test)

In [43]:
df_merged = pd.merge(feature_data, ref_data, on=id_col, how="inner")
print(df_merged.head(), df_merged.shape)

                               sequence  run  VDWAALS       EEL       EGB  \
0  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -236.997 -1869.660  1823.216   
1  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -218.620 -1850.331  1807.831   
2  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -232.611 -1878.075  1834.181   
3  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -203.677 -1870.595  1823.641   
4  CAGGGCTGGGTCCACCTCATGGCCTTTGTTCTGGAA    9 -212.279 -1864.730  1820.462   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
0 -35.292  -2.590101         -156.445725     -4.282747     -24.750849   
1 -32.521  -2.977171         -142.709472     -7.240534     -25.235404   
2 -34.170  -3.105868         -145.088977     -8.856276     -25.124940   
3 -32.402  -3.414769         -150.961716     -5.338670     -23.079573   
4 -31.858  -3.571942         -146.583284     -7.171679     -22.812241   

   bind_avg  
0  0.166339  
1  0.166339  
2  0.166339  
3  0.166339  
4  0.166339   (68040, 11)


In [44]:
# Shuffle everything
df_merged = df_merged.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
print(df_merged.head(), df_merged.shape)

                               sequence  run  VDWAALS       EEL       EGB  \
0  CTGTCCTCTCTCGCCCACGCTGCCTGGGAGGCGCGC   13 -218.518 -1874.250  1828.333   
1  TCCAGTGGCCCCAGGAGGGCCTGGGTGCTCCTCTGC    5 -206.346 -1903.252  1855.652   
2  CCCGGCGAGTCCCGGCCACCCGGCGCAGCCCTGGGC    4 -204.918 -1925.930  1875.508   
3  TCCCCTTGGGAAGAGCACGCGGCGTTGCCAGGCCAA    4 -202.691 -1903.150  1853.510   
4  TAAGGGGTGACCCAGCCGCTGCAGAGCCAGGGAAGG   18 -188.343 -1877.548  1827.619   

    ESURF  HB Energy  Hydrophobic Energy  Pi-Pi Energy  Delta_Entropy  \
0 -35.870  -4.657117         -153.393778     -2.858040     -24.935912   
1 -30.515 -15.204074         -128.334209     -4.376677     -19.790719   
2 -31.144 -15.111787         -137.673951     -0.850311     -22.524782   
3 -32.793  -3.199080         -142.051303     -0.926241     -22.230377   
4 -29.597  -4.119040         -140.533291     -2.594173     -21.882268   

   bind_avg  
0  0.309031  
1 -0.239510  
2 -0.722719  
3  0.358011  
4 -0.710665   (68040, 11)


In [45]:
# Split train vs test by sequence or stratified group
## for regression, not bin or mclass

from sklearn.model_selection import GroupKFold

unique_seqs = df_merged[id_col].unique() #unique sequences are extracted
np.random.seed(random_state)
np.random.shuffle(unique_seqs) #randomly shuffles unique sequences

n_train = int((1 - test_percentage) * len(unique_seqs)) # Compute train/test split boundary

train_seqs = unique_seqs[:n_train] # First 85% (after shuffle) = training sequence IDs.
test_seqs  = unique_seqs[n_train:]

# Filter the rows accordingly
df_train = df_merged[df_merged[id_col].isin(train_seqs)].copy()
df_test  = df_merged[df_merged[id_col].isin(test_seqs)].copy()

print(df_train.shape, df_test.shape)


(56700, 11) (11340, 11)


In [46]:
if "run" in df_train.columns:
    df_train.drop(columns=["run"], inplace=True, errors="ignore")
if "run" in df_test.columns:
    df_test.drop(columns=["run"], inplace=True, errors="ignore")

print(df_train.shape, df_test.shape)

(56700, 10) (11340, 10)


In [47]:
# Save
train_file = f"reg_trn_final.csv"
test_file  = f"reg_tst_preprocess.csv"

df_train.to_csv(train_file, index=False)
df_test.to_csv(test_file, index=False)

In [48]:
###########################################################################
# 2) PROCESS MODE: "train" or "test"
###########################################################################

In [49]:
# Keep only the last X% if requested
keep_last_percent = 90

"""
Retain only the last keep_percent fraction of rows in each sequence group.
If 'run' column exists, sort by it first.
"""

if "run" in df.columns:
    df_sorted = df.sort_values([seq_col, "run"], kind="mergesort")
else:
    df_sorted = df.copy()
group_sizes = df_sorted.groupby(seq_col)[seq_col].transform("size")
cumcount = df_sorted.groupby(seq_col).cumcount()
n_keep = (group_sizes * (keep_percent / 100.0)).astype(int)
n_keep = n_keep.mask(n_keep < 1, 1)  # ensure at least 1 row if fraction>0
mask = cumcount >= (group_sizes - n_keep)
return df_sorted[mask].reset_index(drop=True)

df_train = keep_last_n_percent(df_train, id_col, keep_last_percent)

NameError: name 'seq_col' is not defined